In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
from pyMyriad import *
from pyMyriad.tabular import flatten, tabulate
from pyMyriad.listing import gt_table, simple_table

In [3]:
df = pd.DataFrame({
  "id": np.arange(1000),
  "Gender": np.random.choice(["M", "F"], 1000),
  "Age": np.random.randint(18, 70, 1000),
  "Income": np.random.normal(50000, 15000, 1000),
  "Country": np.random.choice(["Benin", "Albania"], 1000),
  "Education": np.random.normal(0, 1, 1000),
  "Employed": np.random.choice([0, 1], 1000)
})

In [4]:
mfun = lambda df: np.mean(df.Income)
nfun = lambda df: np.std(df.Income)
efun = lambda df: np.mean(df.Education)
benin_fun =  lambda df: df.Country == 'Benin'
age_40 = lambda df: df.Age > 40
age_60 = lambda df: df.Age > 60

atree = AnalysisTree()\
  .split_by("df.Gender")\
  .summarize_by(m = mfun, n = nfun)\
  .split_by(label = "Benin Y/N", expr = benin_fun)\
  .split_by(label = "Age Group", age40 = age_40, age60 = age_60)\
  .analyze_by(m = mfun, n = nfun, label = "Income analysis")\
  .analyze_by(m = efun, label = "Education analysis")

res = atree.run(df)


# Tabulation with simple_table()

The `simple_table()` function provides a lightweight way to display analysis results as a pandas DataFrame without requiring the great-tables package.

In [5]:
# Basic simple_table - shows only analysis results
simple_table(res)

,_Level_0,_Level_1,_Level_2,_Level_3,_Level_4,_Level_5,Statistic,Value
3,Gender,F,--,--,--,--,m,49441.087286
3,,,--,--,--,--,n,14866.715975
8,,,Benin Y/N,False,Age Group,age40,m,51552.779185
8,,,,,,,n,14380.092381
9,,,,,,,m,-0.146261
11,,,,,,age60,m,53804.809282
11,,,,,,,n,13955.316411
12,,,,,,,m,-0.057187
16,,,,True,,age40,m,47891.503267
16,,,,,,,n,15464.315204


## simple_table with pivot

You can pivot by a split variable to show results across columns:

In [6]:
# Pivot by Gender to show results side-by-side
simple_table(res, by="df.Gender")

pivot_lvl,_Level_0,_Level_1,_Level_2,_Level_3,Statistic,F,M
0,--,--,--,--,m,49441.087286,49461.805673
1,--,--,--,--,n,14866.715975,14471.50656
2,Benin Y/N,False,Age Group,age40,m,-0.146261,-0.007014
3,,,,,m,51552.779185,49414.071221
4,,,,,n,14380.092381,14216.028668
5,,,,age60,m,-0.057187,0.149412
6,,,,,m,53804.809282,49993.783777
7,,,,,n,13955.316411,13897.202971
8,,True,,age40,m,0.065342,0.012186
9,,,,,m,47891.503267,49064.528751


## simple_table with all nodes

Include non-analysis nodes (splits and summaries) by setting `include_non_analysis=True`:

In [13]:
# Show all nodes including splits
simple_table(res, include_non_analysis=True).head(10)

,_Level_0,_Level_1,_Level_2,_Level_3,_Level_4,_Level_5,Statistic,Value
0,--,--,--,--,--,--,None,None
1,Gender,--,--,--,--,--,None,None
2,,F,--,--,--,--,None,None
3,,,--,--,--,--,m,49441.087286
3,,,--,--,--,--,n,14866.715975
4,,,Benin Y/N,--,--,--,None,None
5,,,,False,--,--,None,None
6,,,,,Age Group,--,None,None
7,,,,,,age40,None,None
8,,,,,,,m,51552.779185


## simple_table without duplicate suppression

By default, duplicate values in hierarchy columns are suppressed for cleaner display. You can disable this:

In [12]:
# Show all duplicate values
simple_table(res, suppress_duplicates=False).head()

,_Level_0,_Level_1,_Level_2,_Level_3,_Level_4,_Level_5,Statistic,Value
3,Gender,F,--,--,--,--,m,49441.087286
3,Gender,F,--,--,--,--,n,14866.715975
8,Gender,F,Benin Y/N,False,Age Group,age40,m,51552.779185
8,Gender,F,Benin Y/N,False,Age Group,age40,n,14380.092381
9,Gender,F,Benin Y/N,False,Age Group,age40,m,-0.146261


# Tabulation with gt_table()

The `gt_table()` function creates beautifully formatted tables using the great-tables package. It provides more styling options and is ideal for reports and presentations.

In [19]:
# Basic gt_table with title
gt_table(res, title="Analysis Results").show()

Analysis Results 
 
 
 
 Hierarchy 
 
 Statistic 
 Value 
 
 
 _Level_0 
 _Level_1 
 _Level_2 
 _Level_3 
 _Level_4 
 _Level_5 
 
 
 
 
 Gender 
 F 
 -- 
 -- 
 -- 
 -- 
 m 
 49441.0872857053 
 
 
 
 
 -- 
 -- 
 -- 
 -- 
 n 
 14866.715975060022 
 
 
 
 
 Benin Y/N 
 False 
 Age Group 
 age40 
 m 
 51552.77918509381 
 
 
 
 
 
 
 
 
 n 
 14380.092381497016 
 
 
 
 
 
 
 
 
 m 
 -0.14626087858879616 
 
 
 
 
 
 
 
 age60 
 m 
 53804.80928189263 
 
 
 
 
 
 
 
 
 n 
 13955.316410904737 
 
 
 
 
 
 
 
 
 m 
 -0.05718669437712348 
 
 
 
 
 
 True 
 
 age40 
 m 
 47891.503267031454 
 
 
 
 
 
 
 
 
 n 
 15464.315203885219 
 
 
 
 
 
 
 
 
 m 
 0.06534190704016785 
 
 
 
 
 
 
 
 age60 
 m 
 46313.53043699867 
 
 
 
 
 
 
 
 
 n 
 13267.23330285438 
 
 
 
 
 
 
 
 
 m 
 0.050054701552391716 
 
 
 
 M 
 -- 
 -- 
 -- 
 -- 
 m 
 49461.80567261825 
 
 
 
 
 -- 
 -- 
 -- 
 -- 
 n 
 14471.506560092088 
 
 
 
 
 Benin Y/N 
 False 
 Age Group 
 age40 
 m 
 49414.0712212185 
 
 
 
 
 
 
 
 
 n 
 14216.028668023075 
 
 
 
 
 
 
 
 
 m 
 -0.007013606876905443 
 
 
 
 
 
 
 
 age60 
 m 
 49993.783777413744 
 
 
 
 
 
 
 
 
 n 
 13897.202970921557 
 
 
 
 
 
 
 
 
 m 
 0.14941222831196588 
 
 
 
 
 
 True 
 
 age40 
 m 
 49064.52875125308 
 
 
 
 
 
 
 
 
 n 
 14324.190507689087 
 
 
 
 
 
 
 
 
 m 
 0.012185854150159381 
 
 
 
 
 
 
 
 age60 
 m 
 50000.47364152337 
 
 
 
 
 
 
 
 
 n 
 12927.245390968308 
 
 
 
 
 
 
 
 
 m 
 0.07436137956630288

## gt_table with pivot

Pivot by a split variable:

In [10]:
# Pivot by Gender with custom title and subtitle
gt_table(
    res, 
    by="df.Gender", 
    title="Gender Comparison",
    subtitle="Income and Education Analysis by Gender"
).show()

Gender Comparison 
 
 
 Income and Education Analysis by Gender 
 
 
 
 Hierarchy 
 
 Statistic 
 F 
 M 
 
 
 _Level_0 
 _Level_1 
 _Level_2 
 _Level_3 
 
 
 
 
 -- 
 -- 
 -- 
 -- 
 m 
 49441.0872857053 
 49461.80567261825 
 
 
 -- 
 -- 
 -- 
 -- 
 n 
 14866.715975060022 
 14471.506560092088 
 
 
 Benin Y/N 
 False 
 Age Group 
 age40 
 m 
 -0.14626087858879616 
 -0.007013606876905443 
 
 
 
 
 
 
 m 
 51552.77918509381 
 49414.0712212185 
 
 
 
 
 
 
 n 
 14380.092381497016 
 14216.028668023075 
 
 
 
 
 
 age60 
 m 
 -0.05718669437712348 
 0.14941222831196588 
 
 
 
 
 
 
 m 
 53804.80928189263 
 49993.783777413744 
 
 
 
 
 
 
 n 
 13955.316410904737 
 13897.202970921557 
 
 
 
 True 
 
 age40 
 m 
 0.06534190704016785 
 0.012185854150159381 
 
 
 
 
 
 
 m 
 47891.503267031454 
 49064.52875125308 
 
 
 
 
 
 
 n 
 15464.315203885219 
 14324.190507689087 
 
 
 
 
 
 age60 
 m 
 0.050054701552391716 
 0.07436137956630288 
 
 
 
 
 
 
 m 
 46313.53043699867 
 50000.47364152337 
 
 
 
 
 
 
 n 
 13267.23330285438 
 12927.245390968308

## gt_table with all nodes and custom formatting

Include non-analysis nodes and customize decimal places:

In [11]:
# Show all nodes with 2 decimal places
gt_table(
    res, 
    title="Complete Analysis Tree",
    include_non_analysis=True,
    decimals=2
).show()

Complete Analysis Tree 
 
 
 
 Hierarchy 
 
 Statistic 
 Value 
 
 
 _Level_0 
 _Level_1 
 _Level_2 
 _Level_3 
 _Level_4 
 _Level_5 
 
 
 
 
 -- 
 -- 
 -- 
 -- 
 -- 
 -- 
 
 
 
 
 Gender 
 -- 
 -- 
 -- 
 -- 
 -- 
 
 
 
 
 
 F 
 -- 
 -- 
 -- 
 -- 
 
 
 
 
 
 
 -- 
 -- 
 -- 
 -- 
 m 
 49441.0872857053 
 
 
 
 
 -- 
 -- 
 -- 
 -- 
 n 
 14866.715975060022 
 
 
 
 
 Benin Y/N 
 -- 
 -- 
 -- 
 
 
 
 
 
 
 
 False 
 -- 
 -- 
 
 
 
 
 
 
 
 
 Age Group 
 -- 
 
 
 
 
 
 
 
 
 
 age40 
 
 
 
 
 
 
 
 
 
 
 m 
 51552.77918509381 
 
 
 
 
 
 
 
 
 n 
 14380.092381497016 
 
 
 
 
 
 
 
 
 m 
 -0.14626087858879616 
 
 
 
 
 
 
 
 age60 
 
 
 
 
 
 
 
 
 
 
 m 
 53804.80928189263 
 
 
 
 
 
 
 
 
 n 
 13955.316410904737 
 
 
 
 
 
 
 
 
 m 
 -0.05718669437712348 
 
 
 
 
 
 True 
 -- 
 -- 
 
 
 
 
 
 
 
 
 Age Group 
 -- 
 
 
 
 
 
 
 
 
 
 age40 
 
 
 
 
 
 
 
 
 
 
 m 
 47891.503267031454 
 
 
 
 
 
 
 
 
 n 
 15464.315203885219 
 
 
 
 
 
 
 
 
 m 
 0.06534190704016785 
 
 
 
 
 
 
 
 age60 
 
 
 
 
 
 
 
 
 
 
 m 
 46313.53043699867 
 
 
 
 
 
 
 
 
 n 
 13267.23330285438 
 
 
 
 
 
 
 
 
 m 
 0.050054701552391716 
 
 
 
 M 
 -- 
 -- 
 -- 
 -- 
 
 
 
 
 
 
 -- 
 -- 
 -- 
 -- 
 m 
 49461.80567261825 
 
 
 
 
 -- 
 -- 
 -- 
 -- 
 n 
 14471.506560092088 
 
 
 
 
 Benin Y/N 
 -- 
 -- 
 -- 
 
 
 
 
 
 
 
 False 
 -- 
 -- 
 
 
 
 
 
 
 
 
 Age Group 
 -- 
 
 
 
 
 
 
 
 
 
 age40 
 
 
 
 
 
 
 
 
 
 
 m 
 49414.0712212185 
 
 
 
 
 
 
 
 
 n 
 14216.028668023075 
 
 
 
 
 
 
 
 
 m 
 -0.007013606876905443 
 
 
 
 
 
 
 
 age60 
 
 
 
 
 
 
 
 
 
 
 m 
 49993.783777413744 
 
 
 
 
 
 
 
 
 n 
 13897.202970921557 
 
 
 
 
 
 
 
 
 m 
 0.14941222831196588 
 
 
 
 
 
 True 
 -- 
 -- 
 
 
 
 
 
 
 
 
 Age Group 
 -- 
 
 
 
 
 
 
 
 
 
 age40 
 
 
 
 
 
 
 
 
 
 
 m 
 49064.52875125308 
 
 
 
 
 
 
 
 
 n 
 14324.190507689087 
 
 
 
 
 
 
 
 
 m 
 0.012185854150159381 
 
 
 
 
 
 
 
 age60 
 
 
 
 
 
 
 
 
 
 
 m 
 50000.47364152337 
 
 
 
 
 
 
 
 
 n 
 12927.245390968308 
 
 
 
 
 
 
 
 
 m 
 0.07436137956630288

# Pivoting Statistics as Columns

The `pivot_statistics` parameter allows you to display statistics as columns instead of rows, creating a wider, more compact table format.

In [ ]:
# Pivot statistics into columns - simple_table
simple_table(res, pivot_statistics=True)

## Combining pivot_statistics with by

You can combine both pivots to create a matrix-style table with split groups and statistics as columns:

In [ ]:
# Combine both pivots - shows Gender groups with statistics as columns
simple_table(res, by="df.Gender", pivot_statistics=True)

## gt_table with pivot_statistics

The same feature works with `gt_table()` for beautifully formatted output:

In [ ]:
# gt_table with statistics as columns
gt_table(
    res,
    pivot_statistics=True,
    title="Analysis with Statistics as Columns"
).show()

In [ ]:
# gt_table combining both pivots
gt_table(
    res,
    by="df.Gender",
    pivot_statistics=True,
    title="Gender Comparison - Matrix View",
    subtitle="Each group-statistic combination as a column"
).show()